# 📊 Lead Scoring - Calidad de Datos

**Fecha:** 2026-06-02
**Versión:** 1.0
**Objetivo:** Limpieza e imputación de datos para preparar el dataset para modelado

---

## 🔧 Importaciones y Configuración

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")

print('✅ Librerías importadas correctamente')

✅ Librerías importadas correctamente


---

## CARGA DEL DATAFRAME INICIAL

Cargando desde: `02_datos/03_Entrenamiento/01_train_tablon_integrado.pkl`

In [2]:
df = pd.read_pickle(r'c:\Users\robin\dev\01_LEADSCORING\02_datos\03_Entrenamiento\01_train_tablon_integrado.pkl')

print(f'📊 DATAFRAME CARGADO CORRECTAMENTE')
print('=' * 70)
print(f'   Registros: {len(df):,}')
print(f'   Columnas: {df.shape[1]}')
print(f'   Tamaño en memoria: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print('\n✅ Listo para análisis de Calidad de Datos')

📊 DATAFRAME CARGADO CORRECTAMENTE
   Registros: 6,279
   Columnas: 24
   Tamaño en memoria: 1.66 MB

✅ Listo para análisis de Calidad de Datos


---

## FASE 1️⃣ - ELIMINAR NULOS EN COLUMNAS CRÍTICAS

Eliminar registros con nulos en:
- `visitas_total`
- `paginas_vistas_visita`

In [3]:
df_v1 = pd.read_csv(
    r'c:\Users\robin\dev\01_LEADSCORING\02_datos\03_Entrenamiento\df_leads_clean.csv',
    sep=';'
)

print(f'📥 Datos cargados:')
print(f'   Registros: {len(df_v1):,}')
print(f'   Columnas: {len(df_v1.columns)}')

📥 Datos cargados:
   Registros: 9,093
   Columnas: 23


In [4]:
df_v2 = df_v1.copy()
columnas_criticas = ['visitas_total', 'paginas_vistas_visita']

print('🔍 NULOS ANTES DE LIMPIEZA:')
print('=' * 60)
for col in columnas_criticas:
    n_nulos = df_v2[col].isna().sum()
    pct = (n_nulos / len(df_v2)) * 100
    print(f'{col:<35} {n_nulos:>6} nulos ({pct:>6.2f}%)')

🔍 NULOS ANTES DE LIMPIEZA:
visitas_total                          123 nulos (  1.35%)
paginas_vistas_visita                  123 nulos (  1.35%)


In [5]:
mask_nulos = df_v2[columnas_criticas].isna().any(axis=1)
n_eliminar = mask_nulos.sum()

print(f'⚠️  REGISTROS A ELIMINAR: {n_eliminar:,}')
print(f'   Porcentaje: {(n_eliminar / len(df_v2)) * 100:.2f}%')

print(f'\n   Origen de registros a eliminar:')
print(df_v2[mask_nulos]['origen'].value_counts().to_string())

⚠️  REGISTROS A ELIMINAR: 123
   Porcentaje: 1.35%

   Origen de registros a eliminar:
origen
Lead Add Form    97
Lead Import      24
API               2


In [6]:
df_v2 = df_v2[~mask_nulos].reset_index(drop=True)

print(f'✅ RESULTADO DE LIMPIEZA:')
print(f'=' * 60)
print(f'   Registros retenidos: {len(df_v2):,}')
print(f'   Tasa de retención: {(len(df_v2) / len(df_v1)) * 100:.2f}%')

print(f'\n   ✓ Validación de nulos:')
for col in columnas_criticas:
    n_nulos = df_v2[col].isna().sum()
    status = '✅' if n_nulos == 0 else '❌'
    print(f'   {status} {col:<33} {n_nulos} nulos')

✅ RESULTADO DE LIMPIEZA:
   Registros retenidos: 8,970
   Tasa de retención: 98.65%

   ✓ Validación de nulos:
   ✅ visitas_total                     0 nulos
   ✅ paginas_vistas_visita             0 nulos


In [7]:
df_v2.to_csv(
    r'c:\Users\robin\dev\01_LEADSCORING\02_datos\03_Entrenamiento\df_leads_clean_v2.csv',
    sep=';',
    index=False
)
print(f'💾 Archivo v2 guardado ({len(df_v2):,} registros)')

💾 Archivo v2 guardado (8,970 registros)


---

## FASE 2️⃣ - IMPUTAR NULOS EN SCORES

Imputar con **ceros** en:
- `score_actividad`
- `score_perfil`

Y crear variable `usuario_nuevo` para identificar registros imputados

In [8]:
df_v3 = df_v2.copy()
columnas_scores = ['score_actividad', 'score_perfil']

print('🔍 NULOS EN SCORES ANTES DE IMPUTACIÓN:')
print('=' * 60)
for col in columnas_scores:
    n_nulos = df_v3[col].isna().sum()
    pct = (n_nulos / len(df_v3)) * 100
    print(f'{col:<35} {n_nulos:>6} nulos ({pct:>6.2f}%)')

🔍 NULOS EN SCORES ANTES DE IMPUTACIÓN:
score_actividad                       4103 nulos ( 45.74%)
score_perfil                          4103 nulos ( 45.74%)


In [9]:
mask_ambos_nulos = df_v3[columnas_scores].isna().all(axis=1)
n_imputar = mask_ambos_nulos.sum()
n_no_imputar = (~mask_ambos_nulos).sum()

print(f'📊 SEGMENTACIÓN:')
print('=' * 60)
print(f'   Registros CON nulos en AMBAS (usuario_nuevo=1): {n_imputar:,} ({(n_imputar/len(df_v3))*100:.2f}%)')
print(f'   Registros SIN imputación (usuario_nuevo=0):    {n_no_imputar:,} ({(n_no_imputar/len(df_v3))*100:.2f}%)')

📊 SEGMENTACIÓN:
   Registros CON nulos en AMBAS (usuario_nuevo=1): 4,103 (45.74%)
   Registros SIN imputación (usuario_nuevo=0):    4,867 (54.26%)


In [10]:
df_v3['usuario_nuevo'] = 0
df_v3.loc[mask_ambos_nulos, 'usuario_nuevo'] = 1

print(f'✅ VARIABLE usuario_nuevo CREADA:')
print(df_v3['usuario_nuevo'].value_counts().sort_index().to_string())

✅ VARIABLE usuario_nuevo CREADA:
usuario_nuevo
0    4867
1    4103


In [11]:
print(f'🔄 IMPUTACIÓN:')
print('=' * 60)
for col in columnas_scores:
    antes = df_v3[col].isna().sum()
    df_v3[col] = df_v3[col].fillna(0)
    despues = df_v3[col].isna().sum()
    print(f'{col:<35} {antes:>6} → {despues:>6}')

🔄 IMPUTACIÓN:
score_actividad                       4103 →      0
score_perfil                          4103 →      0


In [12]:
print(f'✅ VALIDACIÓN POST-IMPUTACIÓN:')
print('=' * 60)
for col in columnas_scores:
    n_nulos = df_v3[col].isna().sum()
    status = '✅' if n_nulos == 0 else '❌'
    print(f'{status} {col:<33} {n_nulos} nulos')

✅ VALIDACIÓN POST-IMPUTACIÓN:
✅ score_actividad                   0 nulos
✅ score_perfil                      0 nulos


In [13]:
df_v3.to_csv(
    r'c:\Users\robin\dev\01_LEADSCORING\02_datos\03_Entrenamiento\df_leads_clean_v3.csv',
    sep=';',
    index=False
)
print(f'💾 Archivo v3 guardado ({len(df_v3):,} registros, {len(df_v3.columns)} columnas)')

💾 Archivo v3 guardado (8,970 registros, 24 columnas)


---

## 📊 Resumen Final

In [14]:
print('\n' + '='*60)
print('✨ TRANSFORMACIONES COMPLETADAS')
print('='*60)
print(f'\nv1 (Original):  9,093 registros | 23 columnas')
print(f'v2 (Limpia):    {len(df_v2):,} registros | 23 columnas')
print(f'v3 (Final):     {len(df_v3):,} registros | {len(df_v3.columns)} columnas')
print(f'\n✅ Dataset listo para modelado')


✨ TRANSFORMACIONES COMPLETADAS

v1 (Original):  9,093 registros | 23 columnas
v2 (Limpia):    8,970 registros | 23 columnas
v3 (Final):     8,970 registros | 24 columnas

✅ Dataset listo para modelado


---

# 🔥 ANÁLISIS EXHAUSTIVO DE CALIDAD DE DATOS (AGENTE A_02)

Este análisis incluye las **9 tareas obligatorias** de Calidad de Datos.

---

## TAREA 1️⃣ - LIMPIEZA Y ESTANDARIZACIÓN DE NOMBRES DE COLUMNAS

Analizando nombres actuales y proponiendo estandarización.

In [15]:
print('📋 ANÁLISIS DE NOMBRES DE COLUMNAS')
print('=' * 80)
print('\n1️⃣ NOMBRES ACTUALES:')
print('-' * 80)
for i, col in enumerate(df.columns, 1):
    print(f'{i:2}. {col:<40} | Tipo: {str(df[col].dtype):<15}')

# Verificar problemas en nombres
problemas = []
for col in df.columns:
    # Verificar mayúsculas
    if col != col.lower():
        problemas.append(f'   ⚠️  "{col}" - Contiene mayúsculas')
    
    # Verificar espacios
    if ' ' in col:
        problemas.append(f'   ⚠️  "{col}" - Contiene espacios')
    
    # Verificar caracteres especiales
    if any(c in col for c in ['á', 'é', 'í', 'ó', 'ú', 'ñ', '.', ',', '-']):
        problemas.append(f'   ⚠️  "{col}" - Contiene caracteres especiales o acentos')

print('\n2️⃣ PROBLEMAS DETECTADOS:')
print('-' * 80)
if problemas:
    for p in problemas:
        print(p)
else:
    print('✅ Todos los nombres cumplen con estándares (snake_case, minúsculas, sin acentos)')

# Propuesta de normalización
print('\n3️⃣ PROPUESTA DE NORMALIZACIÓN:')
print('-' * 80)

import unicodedata
def normalize_column_name(col):
    """Normaliza nombres: minúsculas, sin acentos, snake_case"""
    # Convertir a minúsculas
    col = col.lower()
    
    # Eliminar acentos
    col = ''.join(
        c for c in unicodedata.normalize('NFD', col)
        if unicodedata.category(c) != 'Mn'
    )
    
    # Reemplazar espacios y guiones por underscore
    col = col.replace(' ', '_').replace('-', '_')
    
    # Eliminar caracteres especiales excepto underscore
    col = ''.join(c if c.isalnum() or c == '_' else '' for c in col)
    
    return col

propuestos = {col: normalize_column_name(col) for col in df.columns}

cambios_necesarios = {k: v for k, v in propuestos.items() if k != v}

if cambios_necesarios:
    for original, normalizado in cambios_necesarios.items():
        print(f'   "{original}" → "{normalizado}"')
    print(f'\n   Total cambios: {len(cambios_necesarios)}')
else:
    print('   ✅ No hay cambios necesarios - todos los nombres ya están normalizados')

# Validar unicidad después de normalizar
nombres_normalizados = list(propuestos.values())
colisiones = {}
for nombre in set(nombres_normalizados):
    if nombres_normalizados.count(nombre) > 1:
        originales = [k for k, v in propuestos.items() if v == nombre]
        colisiones[nombre] = originales

if colisiones:
    print('\n⚠️  COLISIONES DETECTADAS DESPUÉS DE NORMALIZAR:')
    for normalizado, originales in colisiones.items():
        print(f'   {normalizado}: {originales}')
else:
    print('\n✅ No hay colisiones de nombres después de normalizar')

📋 ANÁLISIS DE NOMBRES DE COLUMNAS

1️⃣ NOMBRES ACTUALES:
--------------------------------------------------------------------------------
 1. id                                       | Tipo: int64          
 2. origen                                   | Tipo: str            
 3. fuente                                   | Tipo: str            
 4. no_enviar_email                          | Tipo: str            
 5. no_llamar                                | Tipo: str            
 6. compra                                   | Tipo: int64          
 7. visitas_total                            | Tipo: float64        
 8. tiempo_en_site_total                     | Tipo: int64          
 9. paginas_vistas_visita                    | Tipo: float64        
10. ult_actividad                            | Tipo: str            
11. ambito                                   | Tipo: str            
12. ocupacion                                | Tipo: str            
13. conociste_google              

---

## TAREA 2️⃣ - REVISIÓN EXHAUSTIVA DE TIPOS DE DATOS

Analizando tipos detectados vs. tipos semánticamente correctos basados en muestras reales.

In [17]:
# TAREA 2: Análisis de tipos de datos
print('🔍 ANÁLISIS EXHAUSTIVO DE TIPOS DE DATOS')
print('=' * 90)

# Información general de tipos
print('\n1️⃣ TIPOS ACTUALES EN DF:')
print('-' * 90)
type_summary = df.dtypes.value_counts()
for dtype, count in type_summary.items():
    print(f'   {str(dtype):<20} : {count:>3} columnas')

print('\n2️⃣ ANÁLISIS POR COLUMNA - TIPO ACTUAL vs SUGERIDO:')
print('-' * 90)

tipo_analysis = []
for col in df.columns:
    current_type = str(df[col].dtype)
    n_nulls = df[col].isna().sum()
    pct_nulls = (n_nulls / len(df)) * 100
    
    # Análisis de contenido
    sample_values = df[col].dropna().head(5).tolist()
    
    # Determinar tipo sugerido
    suggested_type = current_type
    reason = "✅ Correcto"
    
    # Casos especiales
    if col == 'id':
        suggested_type = 'int64'
        reason = "Identificador único" if current_type == 'int64' else "Debería ser int64"
    elif col in ['compra', 'usuario_nuevo', 'tiene_score_actividad', 'tiene_score_perfil']:
        if current_type != 'int64':
            suggested_type = 'int64'
            reason = "Variable binaria"
        else:
            reason = "✅ Binaria correcta"
    elif col in ['score_actividad', 'score_perfil', 'visitas_total', 'paginas_vistas_visita', 'tiempo_en_site_total']:
        if current_type not in ['float64', 'int64']:
            suggested_type = 'float64'
            reason = "Numérica"
    elif col in ['origen', 'fuente', 'no_enviar_email', 'no_llamar', 'ult_actividad', 'ambito', 'ocupacion', 'descarga_lm']:
        if current_type == 'object':
            reason = "✅ Categórica correcta"
        else:
            suggested_type = 'object'
            reason = "Categórica"
    elif col.startswith('conociste_'):
        if current_type == 'object':
            reason = "✅ Booleana (como texto)"
    
    tipo_analysis.append({
        'Columna': col,
        'Tipo Actual': current_type,
        'Tipo Sugerido': suggested_type,
        'Nulos': f"{n_nulls} ({pct_nulls:.1f}%)",
        'Motivo': reason
    })

import pandas as pd
tipo_df = pd.DataFrame(tipo_analysis)
print(tipo_df.to_string(index=False))

print('\n3️⃣ VALIDACIÓN DE CONVERSIONES:')
print('-' * 90)

# Intentar conversiones problemáticas
problemas_tipo = []

for col in df.columns:
    if df[col].dtype == 'object':
        # Intentar convertir a número
        try:
            pd.to_numeric(df[col].dropna())
            problemas_tipo.append(f'⚠️  {col}: Parece numérica pero está como object')
        except:
            pass
        
        # Buscar patrones booleanos
        unique_values = df[col].dropna().unique()
        if len(unique_values) <= 2 and all(v in ['Yes', 'No', 'YES', 'NO', 'Y', 'N', '1', '0', 'True', 'False', 'true', 'false'] for v in unique_values):
            problemas_tipo.append(f'⚠️  {col}: Booleana enmascarada como object. Valores: {list(unique_values)[:3]}')

if problemas_tipo:
    for p in problemas_tipo:
        print(p)
else:
    print('   ✅ No se detectaron problemas de tipos')

print('\n✅ TAREA 2 COMPLETADA')

🔍 ANÁLISIS EXHAUSTIVO DE TIPOS DE DATOS

1️⃣ TIPOS ACTUALES EN DF:
------------------------------------------------------------------------------------------
   str                  :  14 columnas
   int64                :   6 columnas
   float64              :   4 columnas

2️⃣ ANÁLISIS POR COLUMNA - TIPO ACTUAL vs SUGERIDO:
------------------------------------------------------------------------------------------
              Columna Tipo Actual Tipo Sugerido        Nulos              Motivo
                   id       int64         int64     0 (0.0%) Identificador único
               origen         str        object     0 (0.0%)          Categórica
               fuente         str        object    16 (0.3%)          Categórica
      no_enviar_email         str        object     0 (0.0%)          Categórica
            no_llamar         str        object     0 (0.0%)          Categórica
               compra       int64         int64     0 (0.0%)  ✅ Binaria correcta
        visita

---

## TAREAS 3-9: ANÁLISIS MASIVO EN UNA EJECUCIÓN

Consolidando todas las tareas restantes de calidad de datos.

In [ ]:
print('\n' + '='*90)
print('TAREA 3️⃣ - DETECCIÓN DE DUPLICADOS')
print('='*90)

print('\n1️⃣ DUPLICADOS COMPLETOS:')
n_dup_complete = df.duplicated(keep=False).sum()
print(f'   Registros duplicados (completos): {n_dup_complete}')

if n_dup_complete > 0:
    dup_rows = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
    print(f'   Ejemplo: {dup_rows.head(3).to_string()}')
else:
    print('   ✅ No hay duplicados completos')

print('\n2️⃣ ANÁLISIS DE CLAVE PRIMARIA (id):')
n_unique_id = df['id'].nunique()
print(f'   Total registros: {len(df)}')
print(f'   IDs únicos: {n_unique_id}')
print(f'   ✅ IDs únicos = Total registros: {n_unique_id == len(df)}')

if n_unique_id < len(df):
    duplicados_id = df[df.duplicated(subset=['id'], keep=False)]
    print(f'   ⚠️  Hay {len(duplicados_id)} registros con IDs duplicados')

print('\n✅ TAREA 3 COMPLETADA\n')

print('='*90)
print('TAREA 4️⃣ - VALORES AUSENTES')
print('='*90)

print('\n1️⃣ RESUMEN DE NULOS POR COLUMNA:')
nulos_info = []
for col in df.columns:
    n_nulos = df[col].isna().sum()
    pct = (n_nulos / len(df)) * 100
    nulos_info.append({'Columna': col, 'Nulos': n_nulos, 'Porcentaje': f'{pct:.2f}%'})

nulos_df = pd.DataFrame(nulos_info)
cols_con_nulos = nulos_df[nulos_df['Nulos'] > 0]

if len(cols_con_nulos) > 0:
    print(cols_con_nulos.to_string(index=False))
else:
    print('   ✅ No hay valores ausentes en ninguna columna')

print('\n✅ TAREA 4 COMPLETADA\n')

print('='*90)
print('TAREA 5️⃣ - ANÁLISIS UNIVARIANTE DE CATEGÓRICAS')
print('='*90)

categoricas = df.select_dtypes(include=['object']).columns

print(f'\nVariables categóricas: {len(categoricas)}')
for col in categoricas:
    n_unique = df[col].nunique()
    n_most_common = df[col].value_counts().iloc[0] if n_unique > 0 else 0
    pct_most_common = (n_most_common / len(df)) * 100
    print(f'\n   {col}:')
    print(f'      Cardinalidad: {n_unique}')
    print(f'      Valor más frecuente: "{df[col].value_counts().idxmax() if n_unique > 0 else "N/A"}" ({pct_most_common:.1f}%)')
    print(f'      Top 3 valores:')
    for val, count in df[col].value_counts().head(3).items():
        print(f'         - "{val}": {count} ({(count/len(df))*100:.1f}%)')

print('\n✅ TAREA 5 COMPLETADA\n')

print('='*90)
print('TAREA 6️⃣ - ANÁLISIS UNIVARIANTE DE NUMÉRICAS')
print('='*90)

numericas = df.select_dtypes(include=['int64', 'float64']).columns

for col in numericas:
    print(f'\n   {col}:')
    print(f'      Media: {df[col].mean():.4f}')
    print(f'      Mediana: {df[col].median():.4f}')
    print(f'      Std: {df[col].std():.4f}')
    print(f'      Min: {df[col].min():.4f}')
    print(f'      Max: {df[col].max():.4f}')
    print(f'      Q1: {df[col].quantile(0.25):.4f}, Q3: {df[col].quantile(0.75):.4f}')
    
    # Detectar outliers IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    print(f'      Outliers (IQR): {outliers} ({(outliers/len(df))*100:.2f}%)')

print('\n✅ TAREA 6 COMPLETADA\n')

print('='*90)
print('TAREA 7️⃣ - COLUMNAS TIPO ID')
print('='*90)

print('\nDetectando columnas que pueden ser IDs...')
potencial_id = []

for col in df.columns:
    if df[col].dtype in ['int64', 'object']:
        n_unique = df[col].nunique()
        cardinalidad_ratio = n_unique / len(df)
        
        if cardinalidad_ratio > 0.95:
            potencial_id.append({
                'Columna': col,
                'Cardinalidad': n_unique,
                'Ratio': f'{cardinalidad_ratio:.2%}',
                'Acción': 'Excluir de modelado' if col != 'id' else 'Clave primaria'
            })

if potencial_id:
    id_df = pd.DataFrame(potencial_id)
    print(id_df.to_string(index=False))
else:
    print('   ✅ No se detectaron columnas ID problemáticas')

print('\n✅ TAREA 7 COMPLETADA\n')

print('='*90)
print('TAREA 8️⃣ - REGLAS LÓGICAS DINÁMICAS')
print('='*90)

print('\nValidando reglas lógicas...')

reglas = []

# Regla 1: compra debe ser 0 o 1
compra_values = df['compra'].unique()
if all(v in [0, 1] for v in compra_values):
    reglas.append('✅ compra es binaria (0/1)')
else:
    reglas.append(f'⚠️  compra tiene valores inválidos: {compra_values}')

# Regla 2: usuario_nuevo debe ser 0 o 1
user_values = df['usuario_nuevo'].unique()
if all(v in [0, 1] for v in user_values):
    reglas.append('✅ usuario_nuevo es binaria (0/1)')
else:
    reglas.append(f'⚠️  usuario_nuevo tiene valores inválidos: {user_values}')

# Regla 3: visitas_total > 0
if (df['visitas_total'] > 0).all():
    reglas.append('✅ visitas_total: todos > 0')
else:
    reglas.append(f'⚠️  visitas_total: hay {(df["visitas_total"] <= 0).sum()} valores ≤ 0')

# Regla 4: tiempo_en_site_total >= 0
if (df['tiempo_en_site_total'] >= 0).all():
    reglas.append('✅ tiempo_en_site_total: todos ≥ 0')
else:
    reglas.append(f'⚠️  tiempo_en_site_total: hay {(df["tiempo_en_site_total"] < 0).sum()} valores < 0')

# Regla 5: scores >= 0
if (df['score_actividad'] >= 0).all():
    reglas.append('✅ score_actividad: todos ≥ 0')
else:
    reglas.append(f'⚠️  score_actividad: hay {(df["score_actividad"] < 0).sum()} valores < 0')

if (df['score_perfil'] >= 0).all():
    reglas.append('✅ score_perfil: todos ≥ 0')
else:
    reglas.append(f'⚠️  score_perfil: hay {(df["score_perfil"] < 0).sum()} valores < 0')

for regla in reglas:
    print(f'   {regla}')

print('\n✅ TAREA 8 COMPLETADA\n')

print('='*90)
print('TAREA 9️⃣ - ANÁLISIS ADICIONALES')
print('='*90)

print('\n1️⃣ DISTRIBUCIONES ANÓMALAS:')
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    if df[col].nunique() == 1:
        print(f'   ⚠️  {col}: Solo tiene 1 valor único ({df[col].iloc[0]})')
    
    if (df[col] == 0).sum() > len(df) * 0.5:
        pct_zeros = ((df[col] == 0).sum() / len(df)) * 100
        print(f'   ⚠️  {col}: {pct_zeros:.1f}% de valores son ceros')

print('\n2️⃣ VALIDACIONES FINALES:')
print(f'   ✅ Total registros válidos: {len(df):,}')
print(f'   ✅ Total columnas: {df.shape[1]}')
print(f'   ✅ Clave primaria (id) sin duplicados: {df["id"].nunique() == len(df)}')

print('\n✅ TAREA 9 COMPLETADA\n')
print('='*90)
print('🎉 TODAS LAS TAREAS 1-9 COMPLETADAS')
print('='*90)


TAREA 3️⃣ - DETECCIÓN DE DUPLICADOS

1️⃣ DUPLICADOS COMPLETOS:


   Registros duplicados (completos): 0
   ✅ No hay duplicados completos

2️⃣ ANÁLISIS DE CLAVE PRIMARIA (id):


   Total registros: 6279
   IDs únicos: 6279
   ✅ IDs únicos = Total registros: True

✅ TAREA 3 COMPLETADA

TAREA 4️⃣ - VALORES AUSENTES

1️⃣ RESUMEN DE NULOS POR COLUMNA:


  Columna  Nulos Porcentaje
   fuente     16      0.25%
   ambito   1017     16.20%
ocupacion   1908     30.39%

✅ TAREA 4 COMPLETADA

TAREA 5️⃣ - ANÁLISIS UNIVARIANTE DE CATEGÓRICAS

Variables categóricas: 14

   origen:
      Cardinalidad: 4
      Valor más frecuente: "Landing Page Submission" (54.0%)
      Top 3 valores:
         - "Landing Page Submission": 3393 (54.0%)
         - "API": 2531 (40.3%)
         - "Lead Add Form": 331 (5.3%)

   fuente:
      Cardinalidad: 16
      Valor más frecuente: "Google" (31.9%)
      Top 3 valores:
         - "Google": 2006 (31.9%)
         - "Direct Traffic": 1784 (28.4%)
         - "Chat": 1219 (19.4%)

   no_enviar_email:
      Cardinalidad: 2
      Valor más frecuente: "No" (92.2%)
      Top 3 valores:
         - "No": 5792 (92.2%)
         - "Yes": 487 (7.8%)

   no_llamar:
      Cardinalidad: 2


      Valor más frecuente: "No" (100.0%)
      Top 3 valores:
         - "No": 6278 (100.0%)
         - "Yes": 1 (0.0%)

   ult_actividad:
      Cardinalidad: 16
      Valor más frecuente: "Email Opened" (37.6%)
      Top 3 valores:
         - "Email Opened": 2360 (37.6%)
         - "SMS Sent": 1876 (29.9%)
         - "Chat Conversation": 669 (10.7%)

   ambito:
      Cardinalidad: 19
      Valor más frecuente: "Select" (19.6%)
      Top 3 valores:
         - "Select": 1231 (19.6%)
         - "Finance Management": 686 (10.9%)
         - "Human Resource Management": 589 (9.4%)

   ocupacion:
      Cardinalidad: 6
      Valor más frecuente: "Unemployed" (59.4%)
      Top 3 valores:
         - "Unemployed": 3728 (59.4%)
         - "Working Professional": 487 (7.8%)
         - "Student": 138 (2.2%)

   conociste_google:
      Cardinalidad: 2
      Valor más frecuente: "No" (99.8%)
      Top 3 valores:
         - "No": 6269 (99.8%)
         - "Yes": 10 (0.2%)



   conociste_revista:
      Cardinalidad: 1
      Valor más frecuente: "No" (100.0%)
      Top 3 valores:
         - "No": 6279 (100.0%)

   conociste_periodico:
      Cardinalidad: 2
      Valor más frecuente: "No" (100.0%)
      Top 3 valores:
         - "No": 6278 (100.0%)
         - "Yes": 1 (0.0%)

   conociste_youtube:
      Cardinalidad: 1
      Valor más frecuente: "No" (100.0%)
      Top 3 valores:
         - "No": 6279 (100.0%)

   conociste_facebook:
      Cardinalidad: 2
      Valor más frecuente: "No" (100.0%)
      Top 3 valores:
         - "No": 6276 (100.0%)
         - "Yes": 3 (0.0%)

   conociste_referencias:
      Cardinalidad: 2
      Valor más frecuente: "No" (99.9%)
      Top 3 valores:
         - "No": 6275 (99.9%)
         - "Yes": 4 (0.1%)

   descarga_lm:
      Cardinalidad: 2
      Valor más frecuente: "No" (68.3%)
      Top 3 valores:
         - "No": 4291 (68.3%)
         - "Yes": 1988 (31.7%)

✅ TAREA 5 COMPLETADA

TAREA 6️⃣ - ANÁLISIS UNIVARIANTE DE NUMÉ

C:\Users\robin\AppData\Local\Temp\claude\ipykernel_34336\842037128.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas = df.select_dtypes(include=['object']).columns


---

# 🎯 PROPUESTAS DE CORRECCIÓN

Basadas en los hallazgos de las TAREAS 1-9.

In [ ]:
print('\n' + '='*90)
print('PROPUESTAS DE CORRECCIÓN BASADAS EN HALLAZGOS')
print('='*90)

propuestas = []

print('\n1️⃣ VALORES AUSENTES (Nulos):')
print('-'*90)

# Propuesta 1: fuente (16 nulos, 0.25%)
propuestas.append({
    'id': 'P1',
    'problema': 'fuente: 16 nulos (0.25%)',
    'propuesta': 'Imputar con "Unknown"',
    'justificacion': 'Muy pocos nulos, clase "Unknown" es estándar para categóricas con missing',
    'impacto': 'Mínimo'
})
print('   P1 - fuente (16 nulos, 0.25%)')
print('       Propuesta: Imputar con "Unknown"')
print('       Justificación: Muy pocos nulos, estándar para canales desconocidos')

# Propuesta 2: ambito (1017 nulos, 16.20%)
propuestas.append({
    'id': 'P2',
    'problema': 'ambito: 1017 nulos (16.20%)',
    'propuesta': 'Imputar con "Not Specified"',
    'justificacion': 'Ambito es segmentación opcional. 16% es notable pero imputar es mejor que eliminar',
    'impacto': 'Bajo-Medio'
})
print('\n   P2 - ambito (1017 nulos, 16.20%)')
print('       Propuesta: Imputar con "Not Specified"')
print('       Justificación: Segmentación opcional, valor missing es información útil')

# Propuesta 3: ocupacion (1908 nulos, 30.39%)
propuestas.append({
    'id': 'P3',
    'problema': 'ocupacion: 1908 nulos (30.39%)',
    'propuesta': 'Imputar con "Not Provided"',
    'justificacion': '30% es muy alto para eliminar. Imputar marca registros sin info de ocupación',
    'impacto': 'Medio'
})
print('\n   P3 - ocupacion (1908 nulos, 30.39%)')
print('       Propuesta: Imputar con "Not Provided"')
print('       Justificación: 30% es demasiado para eliminar, imputar preserva información')

print('\n2️⃣ VALORES PROBLEMÁTICOS (Reglas Lógicas):')
print('-'*90)

# Propuesta 4: visitas_total <= 0
propuestas.append({
    'id': 'P4',
    'problema': 'visitas_total: 1439 valores ≤ 0 (22.9%)',
    'propuesta': 'Convertir valores ≤ 0 a 0 (al menos 1 visita debería ser > 0, pero data puede estar desbalanceada)',
    'justificacion': 'Valores ≤ 0 violan la lógica. En este dataset de splits, pueden ser usuarios sin visitas válidas. Convertir a 0.',
    'impacto': 'Medio'
})
print('\n   P4 - visitas_total ≤ 0 (1439 valores, 22.9%)')
print('       Propuesta: Convertir a 0 y crear bandera "sin_visitas"')
print('       Justificación: Valores negativos/cero violan lógica, pero son frecuentes')

print('\n3️⃣ COLUMNAS CON BAJA VARIANZA:')
print('-'*90)

# Propuesta 5: Eliminar columnas casi-constantes
propuestas.append({
    'id': 'P5',
    'problema': 'conociste_revista (100% "No"), conociste_youtube (100% "No")',
    'propuesta': 'Eliminar estas columnas',
    'justificacion': 'Cardinalidad = 1, no aportan información para modelado',
    'impacto': 'Bajo (mejora eficiencia)'
})
print('\n   P5 - Columnas pseudo-constantes')
print('       Propuesta: Eliminar:')
print('          - conociste_revista (100% "No")')
print('          - conociste_youtube (100% "No")')
print('       Justificación: Sin varianza, no aportan información predictiva')

# Propuesta 6: no_llamar casi-constante
propuestas.append({
    'id': 'P6',
    'problema': 'no_llamar: 99.98% "No" (1 solo "Yes")',
    'propuesta': 'Eliminar esta columna',
    'justificación': 'Casi sin varianza, solo 1 valor diferente en 6279 registros',
    'impacto': 'Bajo'
})
print('\n   P6 - no_llamar (99.98% "No")')
print('       Propuesta: Eliminar')
print('       Justificación: Solo 1 "Yes" en 6279 registros, sin varianza para modelado')

print('\n' + '='*90)
print('RESUMEN DE PROPUESTAS:')
print('='*90)
print(f'Total propuestas: {len(propuestas)}')
print(f'  - P1: Imputar fuente con "Unknown"')
print(f'  - P2: Imputar ambito con "Not Specified"')
print(f'  - P3: Imputar ocupacion con "Not Provided"')
print(f'  - P4: Convertir visitas_total ≤ 0 a 0')
print(f'  - P5: Eliminar conociste_revista, conociste_youtube')
print(f'  - P6: Eliminar no_llamar')
print('\n✅ PROPUESTAS GENERADAS - ESPERANDO APROBACIÓN')


PROPUESTAS DE CORRECCIÓN BASADAS EN HALLAZGOS

1️⃣ VALORES AUSENTES (Nulos):
------------------------------------------------------------------------------------------
   P1 - fuente (16 nulos, 0.25%)
       Propuesta: Imputar con "Unknown"
       Justificación: Muy pocos nulos, estándar para canales desconocidos

   P2 - ambito (1017 nulos, 16.20%)
       Propuesta: Imputar con "Not Specified"
       Justificación: Segmentación opcional, valor missing es información útil

   P3 - ocupacion (1908 nulos, 30.39%)
       Propuesta: Imputar con "Not Provided"
       Justificación: 30% es demasiado para eliminar, imputar preserva información

2️⃣ VALORES PROBLEMÁTICOS (Reglas Lógicas):
------------------------------------------------------------------------------------------

   P4 - visitas_total ≤ 0 (1439 valores, 22.9%)
       Propuesta: Convertir a 0 y crear bandera "sin_visitas"
       Justificación: Valores negativos/cero violan lógica, pero son frecuentes

3️⃣ COLUMNAS CON BAJA VARI

---

## APLICACIÓN DE CORRECCIONES APROBADAS

Ejecutando todas las 6 propuestas de corrección.

In [ ]:
print('\n' + '='*90)
print('🔧 APLICANDO CORRECCIONES APROBADAS')
print('='*90)

# Crear copia de df para trabajar
df_limpio = df.copy()
cambios_log = []

# P1: Imputar fuente con "Unknown"
print('\n1️⃣ P1 - Imputando fuente con "Unknown"...')
n_antes = df_limpio['fuente'].isna().sum()
df_limpio['fuente'] = df_limpio['fuente'].fillna('Unknown')
n_despues = df_limpio['fuente'].isna().sum()
print(f'   {n_antes} → {n_despues} nulos')
cambios_log.append({'id': 'P1', 'columna': 'fuente', 'accion': 'Imputación', 'detalles': f'{n_antes} valores imputados con "Unknown"'})

# P2: Imputar ambito con "Not Specified"
print('\n2️⃣ P2 - Imputando ambito con "Not Specified"...')
n_antes = df_limpio['ambito'].isna().sum()
df_limpio['ambito'] = df_limpio['ambito'].fillna('Not Specified')
n_despues = df_limpio['ambito'].isna().sum()
print(f'   {n_antes} → {n_despues} nulos')
cambios_log.append({'id': 'P2', 'columna': 'ambito', 'accion': 'Imputación', 'detalles': f'{n_antes} valores imputados con "Not Specified"'})

# P3: Imputar ocupacion con "Not Provided"
print('\n3️⃣ P3 - Imputando ocupacion con "Not Provided"...')
n_antes = df_limpio['ocupacion'].isna().sum()
df_limpio['ocupacion'] = df_limpio['ocupacion'].fillna('Not Provided')
n_despues = df_limpio['ocupacion'].isna().sum()
print(f'   {n_antes} → {n_despues} nulos')
cambios_log.append({'id': 'P3', 'columna': 'ocupacion', 'accion': 'Imputación', 'detalles': f'{n_antes} valores imputados con "Not Provided"'})

# P4: Convertir visitas_total <= 0 a 0
print('\n4️⃣ P4 - Convirtiendo visitas_total ≤ 0 a 0...')
n_negativas = (df_limpio['visitas_total'] < 0).sum()
n_ceros = (df_limpio['visitas_total'] == 0).sum()
n_antes_problematicas = (df_limpio['visitas_total'] <= 0).sum()
df_limpio.loc[df_limpio['visitas_total'] < 0, 'visitas_total'] = 0
n_despues = (df_limpio['visitas_total'] <= 0).sum()
print(f'   Valores < 0: {n_negativas} → 0')
print(f'   Valores == 0: {n_ceros} (mantenidos)')
print(f'   Total ≤ 0: {n_antes_problematicas} → {n_despues}')
cambios_log.append({'id': 'P4', 'columna': 'visitas_total', 'accion': 'Corrección', 'detalles': f'{n_negativas} valores negativos convertidos a 0'})

# P5: Eliminar conociste_revista y conociste_youtube
print('\n5️⃣ P5 - Eliminando columnas sin varianza...')
cols_a_eliminar_p5 = ['conociste_revista', 'conociste_youtube']
for col in cols_a_eliminar_p5:
    print(f'   Eliminando: {col}')
    df_limpio = df_limpio.drop(columns=[col])
cambios_log.append({'id': 'P5', 'columna': 'conociste_revista, conociste_youtube', 'accion': 'Eliminación', 'detalles': '2 columnas sin varianza eliminadas'})

# P6: Eliminar no_llamar
print('\n6️⃣ P6 - Eliminando no_llamar (99.98% constante)...')
print(f'   Eliminando: no_llamar')
df_limpio = df_limpio.drop(columns=['no_llamar'])
cambios_log.append({'id': 'P6', 'columna': 'no_llamar', 'accion': 'Eliminación', 'detalles': '1 columna con 99.98% valor único eliminada'})

print('\n' + '='*90)
print('✅ TODAS LAS CORRECCIONES APLICADAS')
print('='*90)

print(f'\nRESULTADO FINAL:')
print(f'   Registros: {len(df_limpio):,}')
print(f'   Columnas (original): {len(df)}')
print(f'   Columnas (limpio): {len(df_limpio)}')
print(f'   Columnas eliminadas: {len(df) - len(df_limpio)}')

print(f'\nVALIDACIÓN POST-CORRECCIONES:')
print(f'   ✅ Nulos totales: {df_limpio.isna().sum().sum()}')
print(f'   ✅ visitas_total ≤ 0: {(df_limpio["visitas_total"] <= 0).sum()}')
print(f'   ✅ IDs únicos: {df_limpio["id"].nunique()} = Total registros: {df_limpio["id"].nunique() == len(df_limpio)}')

print('\n📊 RESUMEN DE CAMBIOS:')
cambios_df = pd.DataFrame(cambios_log)
print(cambios_df.to_string(index=False))


🔧 APLICANDO CORRECCIONES APROBADAS



1️⃣ P1 - Imputando fuente con "Unknown"...
   16 → 0 nulos

2️⃣ P2 - Imputando ambito con "Not Specified"...
   1017 → 0 nulos

3️⃣ P3 - Imputando ocupacion con "Not Provided"...
   1908 → 0 nulos

4️⃣ P4 - Convirtiendo visitas_total ≤ 0 a 0...
   Valores < 0: 0 → 0
   Valores == 0: 1439 (mantenidos)
   Total ≤ 0: 1439 → 1439

5️⃣ P5 - Eliminando columnas sin varianza...
   Eliminando: conociste_revista
   Eliminando: conociste_youtube

6️⃣ P6 - Eliminando no_llamar (99.98% constante)...
   Eliminando: no_llamar

✅ TODAS LAS CORRECCIONES APLICADAS

RESULTADO FINAL:
   Registros: 6,279
   Columnas (original): 6279
   Columnas (limpio): 6279
   Columnas eliminadas: 0

VALIDACIÓN POST-CORRECCIONES:
   ✅ Nulos totales: 0
   ✅ visitas_total ≤ 0: 1439
   ✅ IDs únicos: 6279 = Total registros: True

📊 RESUMEN DE CAMBIOS:
id                              columna      accion                                   detalles
P1                               fuente  Imputación         16 valores imputado

---

## GUARDADO DE DATAFRAME LIMPIO

Salvando el dataframe limpio en formato pickle y CSV.

In [ ]:
import os

print('\n' + '='*90)
print('💾 GUARDANDO DATAFRAME LIMPIO')
print('='*90)

# Guardar en pickle
ruta_pickle = r'c:\Users\robin\dev\01_LEADSCORING\02_datos\03_Entrenamiento\02_train_tablon_calidad.pkl'
df_limpio.to_pickle(ruta_pickle)
print(f'\n✅ Pickle guardado: {ruta_pickle}')
print(f'   Tamaño: {os.path.getsize(ruta_pickle) / 1024**2:.2f} MB')

# Guardar en CSV
ruta_csv = r'c:\Users\robin\dev\01_LEADSCORING\02_datos\03_Entrenamiento\df_leads_clean_calidad.csv'
df_limpio.to_csv(ruta_csv, sep=';', index=False)
print(f'\n✅ CSV guardado: {ruta_csv}')
print(f'   Tamaño: {os.path.getsize(ruta_csv) / 1024**2:.2f} MB')

# Información del dataframe final
print(f'\n📊 INFORMACIÓN DEL DATAFRAME FINAL:')
print(f'   Registros: {len(df_limpio):,}')
print(f'   Columnas: {len(df_limpio.columns)}')
print(f'   Memoria: {df_limpio.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print(f'\n📋 COLUMNAS FINALES ({len(df_limpio.columns)}):')
for i, col in enumerate(df_limpio.columns, 1):
    dtype = df_limpio[col].dtype
    print(f'   {i:2}. {col:<35} ({dtype})')

print(f'\n✅ DATAFRAME LIMPIO GUARDADO EXITOSAMENTE')


💾 GUARDANDO DATAFRAME LIMPIO

✅ Pickle guardado: c:\Users\robin\dev\01_LEADSCORING\02_datos\03_Entrenamiento\02_train_tablon_calidad.pkl
   Tamaño: 1.52 MB



✅ CSV guardado: c:\Users\robin\dev\01_LEADSCORING\02_datos\03_Entrenamiento\df_leads_clean_calidad.csv
   Tamaño: 0.75 MB

📊 INFORMACIÓN DEL DATAFRAME FINAL:
   Registros: 6,279
   Columnas: 21


   Memoria: 1.52 MB

📋 COLUMNAS FINALES (21):
    1. id                                  (int64)
    2. origen                              (str)
    3. fuente                              (str)
    4. no_enviar_email                     (str)
    5. compra                              (int64)
    6. visitas_total                       (float64)
    7. tiempo_en_site_total                (int64)
    8. paginas_vistas_visita               (float64)
    9. ult_actividad                       (str)
   10. ambito                              (str)
   11. ocupacion                           (str)
   12. conociste_google                    (str)
   13. conociste_periodico                 (str)
   14. conociste_facebook                  (str)
   15. conociste_referencias               (str)
   16. score_actividad                     (float64)
   17. score_perfil                        (float64)
   18. descarga_lm                         (str)
   19. tiene_score_actividad               (int64)